# Lesson 4.4: Collaborative Location Review

The geoparser resolved hundreds of place names from Reddit posts, but automatic geocoders make predictable mistakes:

| Error type | Example |
|---|---|
| Wrong disambiguation | `JMU` → Jiamusi, China instead of James Madison University |
| Wrong coordinates | `Harrisonburg` placed in the wrong county |
| False positive | `Virginia` captured as a destination when it is just context |
| Spurious match | `D-Hall` → a mall in New York |

Your team will work through these locations together in Google Sheets, then export a clean file for Lessons 5 and 6.

**Workflow overview**

| Part | Tool | Who |
|---|---|---|
| A — Prepare the review file | Python (this notebook) | One student |
| B — Review locations | Google Sheets | Whole team |
| C — Load and verify from Google Sheets | Python (this notebook) | One student |
| D — Export the cleaned file | Python (this notebook) | One student |
| E — Commit to Git | Codespaces | One student |


## Part A: Load the Review File

Run the cell below to confirm the file is ready. The six review columns (`action`, `corrected_name`, `corrected_lat`, `corrected_lon`, `corrected_place_type`, `reviewer`) are already included — they were added by the pipeline when the data was generated.


In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('../data/JMU/JMU_geoparsed_long.csv')

print(f"✅ Loaded: {len(df):,} rows  |  {len(df.columns)} columns")
print(f"\nplace_type distribution:")
print(df['place_type'].value_counts(dropna=False).to_string())
print(f"\nNext: open  ../data/JMU/JMU_geoparsed_long.csv  in Google Sheets → follow Part B.")


✅ Loaded: 884 rows  |  20 columns

Next: open  data/jmu_reddit_geoparsed_long.csv  in Google Sheets → follow Part B.


,type,date,score,sentences,year_month,place,latitude,longitude,feature_type,admin1_name,admin2_name,country_name,school,place_type,action,corrected_name,corrected_lat,corrected_lon,corrected_place_type,reviewer
0,comment,2025-12-22,36,I’m in the band and got to go on this trip to ...,2025-12,Oregon,44.00013,-120.50139,first-order administrative division,Oregon,NaN,United States,JMU,State,NaN,NaN,NaN,NaN,NaN,NaN
1,comment,2025-12-22,13,"I have never, ever, met a nicer community than...",2025-12,Eugene,44.05207,-123.08675,seat of a second-order administrative division,Oregon,Lane County,United States,JMU,City,NaN,NaN,NaN,NaN,NaN,NaN
2,comment,2025-12-22,13,"I have never, ever, met a nicer community than...",2025-12,Oregon,44.00013,-120.50139,first-order administrative division,Oregon,NaN,United States,JMU,State,NaN,NaN,NaN,NaN,NaN,NaN
3,comment,2025-12-23,3,I got to perform in Monaco and the French Rivi...,2025-12,Principality of Monaco,43.73141,7.41903,independent political entity,Municipality of Monaco,NaN,Monaco,JMU,Country,NaN,NaN,NaN,NaN,NaN,NaN
4,comment,2025-12-22,-12,In contrast the Oregon band looked like Uncle ...,2025-12,Oregon,44.00013,-120.50139,first-order administrative division,Oregon,NaN,United States,JMU,State,NaN,NaN,NaN,NaN,NaN,NaN


## Part B: Review in Google Sheets

**One student** starts this step; the whole team contributes.

### B1 — Import the file

1. Go to [sheets.google.com](https://sheets.google.com) → **Blank spreadsheet**
2. **File → Import → Upload** → select `data/JMU/JMU_geoparsed_long.csv` from the repo root
3. Choose **Replace spreadsheet**, separator type **Comma**
4. Rename the spreadsheet: `JMU_geoparsed_long_cleaned`
5. **Share → Anyone with the link → Editor** → copy the link and post it in your team channel

> 💡 **Tip:** Sort by `place_count` descending first — high-frequency places matter most and are worth checking carefully.

### B2 — Add data validation

Set up three dropdown validations so the whole team fills in consistent values.

**Column `action`** *(most important)*:

1. Click the `action` column header to select the whole column
2. **Data → Data validation → Add rule**
3. Criteria: **Dropdown** → add three options: `KEEP`, `CORRECT`, `REMOVE`
4. "If data is invalid": **Reject input**
5. Right-click the `action` header cell → **Insert note** → paste: *KEEP = location is correct. CORRECT = right place, wrong details. REMOVE = not a real location or geoparser error.*

**Column `reviewer`**:

1. Select the `reviewer` column → **Data → Data validation → Add rule**
2. Criteria: **Dropdown** → add each team member's name

**Column `corrected_place_type`**:

1. Select the `corrected_place_type` column → **Data → Data validation → Add rule**
2. Criteria: **Dropdown** → add: `Country`, `State`, `Region`, `City`, `Neighborhood`, `University`, `Road`, `Building`, `Natural Feature`

### B3 — Review the rows

Work through the rows as a team. For each location:

- Read the `sentences` column to understand the context
- Check `place`, `latitude`, `longitude`, and `place_type`
- Set `action` to `KEEP`, `CORRECT`, or `REMOVE`
- If `CORRECT`: fill in only the columns that need changing:
  - `corrected_name` — new place name
  - `corrected_latlon` — paste directly from Google Maps (e.g. `38.433998, -78.872973`)
  - `corrected_place_type` — select from the dropdown
- Enter your name in `reviewer`

> 💡 **Getting coordinates from Google Maps:** Right-click any spot on the map → click the coordinates at the top of the menu → they copy automatically. Paste the full string (`38.433998, -78.872973`) into `corrected_latlon` — the comma is fine, Python will split it on import.

### B4 — Publish the sheet

Once your team has reviewed all rows:

1. **File → Share → Publish to web**
2. Choose: **Entire document** → **Comma-separated values (.csv)**
3. Click **Publish** → copy the URL
4. Paste the URL into the `SHEETS_LINK` variable in **Part C** below


## Part C: Load from Google Sheets and Verify

Paste your published sheet URL into `SHEETS_LINK` in the cell below and run both cells. The map lets you visually check for any remaining misplaced pins before you export.


In [ ]:
# ← Paste your published Google Sheets CSV link here
SHEETS_LINK = "https://docs.google.com/spreadsheets/d/PASTE_YOUR_LINK_HERE/pub?output=csv"

df_review = pd.read_csv(SHEETS_LINK)

# Split the "lat, lon" Google Maps paste string into separate columns
if 'corrected_latlon' in df_review.columns:
    coords = df_review['corrected_latlon'].astype(str).str.split(',', n=1, expand=True)
    df_review['corrected_lat'] = coords[0].str.strip().replace({'nan': '', 'None': ''})
    df_review['corrected_lon'] = (coords[1].str.strip().replace({'nan': '', 'None': ''})
                                  if coords.shape[1] > 1 else '')

print(f"✅ Loaded {len(df_review):,} rows from Google Sheets")
print(f"\nReview progress:")
print(df_review['action'].value_counts(dropna=False).to_string())

reviewed = df_review['action'].isin(['KEEP', 'CORRECT', 'REMOVE']).sum()
print(f"\n{reviewed} / {len(df_review)} rows reviewed ({reviewed / len(df_review) * 100:.0f}%)")


✅ Loaded 884 rows from Google Sheets

Review progress:
action
NaN        442
KEEP       224
CORRECT    217
REMOVE       1

442 / 884 rows reviewed (50%)


In [14]:
# Verification map — inspect dots geographically
# Points far outside Virginia / the US East Coast are worth checking

df_map = df_review[
    df_review['action'].isin(['KEEP', 'CORRECT']) |
    df_review['action'].isna() |
    (df_review['action'] == '')
].copy()

df_map['lat_plot'] = pd.to_numeric(
    df_map['corrected_lat'].where(
        df_map['corrected_lat'].notna() & (df_map['corrected_lat'].astype(str).str.strip() != ''),
        df_map['latitude']
    ), errors='coerce')

df_map['lon_plot'] = pd.to_numeric(
    df_map['corrected_lon'].where(
        df_map['corrected_lon'].notna() & (df_map['corrected_lon'].astype(str).str.strip() != ''),
        df_map['longitude']
    ), errors='coerce')

df_map = df_map.dropna(subset=['lat_plot', 'lon_plot'])

fig = px.scatter_map(
    df_map,
    lat='lat_plot', lon='lon_plot',
    hover_name='place',
    hover_data={'place_type': True, 'action': True,
                'lat_plot': False, 'lon_plot': False},
    color='action',
    color_discrete_map={'KEEP': '#2ca02c', 'CORRECT': '#ff7f0e', '': '#aec7e8'},
    size_max=12,
    map_style='carto-positron',
    center={'lat': 37.5, 'lon': -78.0}, zoom=4,
    height=500,
    title='Location review map — hover for details'
)
fig.update_layout(margin=dict(r=0, t=50, l=0, b=0))
fig.show()

print("\n💡 Dots far outside Virginia may be geoparser errors.")
print("   Go back to the sheet and mark them REMOVE if needed, then re-run Part C.")



💡 Dots far outside Virginia may be geoparser errors.
   Go back to the sheet and mark them REMOVE if needed, then re-run Part C.


## Part D: Export the Cleaned File

When your team is satisfied with the review, run the cell below. It will:

- Drop every row marked `REMOVE`
- Apply any corrections (`corrected_name` → `place`, `corrected_lat` → `latitude`, etc.)
- Remove the six review columns
- Save the result as `../data/JMU/JMU_geoparsed_cleaned.csv`

This file feeds directly into Lessons 5 and 6.


In [ ]:
# Apply corrections and export the cleaned file
df_out = df_review.copy()

# Drop rows marked REMOVE
n_before = len(df_out)
df_out = df_out[df_out['action'] != 'REMOVE'].copy()
n_removed = n_before - len(df_out)

# Apply corrections where non-empty
def apply_correction(df, corrected_col, target_col, cast=None):
    mask = df[corrected_col].notna() & (df[corrected_col].astype(str).str.strip() != '')
    if cast:
        df.loc[mask, target_col] = pd.to_numeric(df.loc[mask, corrected_col], errors='coerce')
    else:
        df.loc[mask, target_col] = df.loc[mask, corrected_col]
    return df, mask.sum()

df_out, n_name = apply_correction(df_out, 'corrected_name',       'place')
df_out, n_lat  = apply_correction(df_out, 'corrected_lat',        'latitude',  cast=True)
df_out, n_lon  = apply_correction(df_out, 'corrected_lon',        'longitude', cast=True)
df_out, n_type = apply_correction(df_out, 'corrected_place_type', 'place_type')

# Drop the review columns from the final file
review_cols = ['action', 'corrected_name', 'corrected_latlon', 'corrected_lat', 'corrected_lon',
               'corrected_place_type', 'reviewer', 'place_count']
df_out = df_out.drop(columns=[c for c in review_cols if c in df_out.columns])

df_out.to_csv('../data/JMU/JMU_geoparsed_cleaned.csv', index=False)

print(f"✅ Saved  ../data/JMU/JMU_geoparsed_cleaned.csv")
print(f"\nSummary:")
print(f"  {n_before:,} rows in  →  {len(df_out):,} rows out  ({n_removed} removed)")
print(f"  Names corrected:        {n_name}")
print(f"  Coordinates corrected:  {n_lat} lat  /  {n_lon} lon")
print(f"  Place types corrected:  {n_type}")


## Part E: Commit to Git

**One student on the team** does this step after the export is complete.

### Create a branch and commit

1. Open a terminal (**Terminal → New Terminal** in VS Code)

2. Create a new branch:
   ```
   git checkout -b location-review
   ```

3. Stage and commit the cleaned file:
   ```
   git add data/jmu_reddit_geoparsed_cleaned.csv
   git commit -m "Add reviewed and cleaned geoparsed locations"
   ```

4. Push the branch to GitHub:
   ```
   git push origin location-review
   ```

5. Open the repository on GitHub and create a **Pull Request** from `location-review` → `main`

6. Request a review from a teammate

7. Once approved, click **Merge pull request**

---

> ⚠️ **Do not commit `jmu_reddit_geoparsed_long.csv`** — it contains the raw review columns and is for your team's working use only.

> 💡 If your team did not complete this review in time, a backup cleaned file is provided in `data/` that you can use as a fallback for Lessons 5 and 6.
